# Create a Custom Root-Zone Water Dataset

This notebook is the starting point when you have CDSE credentials and want to
create your own dataset for **Root Zone Water Estimation with Satellite Data**.

The workflow is deliberately two-stage:

1. Enter dates, a filter and a project-relative dataset directory, then generate a new config.
2. Enable the download cell to fetch matching Open-Meteo weather and Sentinel-2 L2A statistics.

The acquisition saves raw responses, quality flags, catalog metadata, geometry,
settings and checksums. The estimation notebook can then load the saved directory
without credentials or network access. Existing datasets are never overwritten.

## Before downloading

Create a CDSE OAuth client and place its two values in a private environment file.
The file must define `SENTINEL_CLIENT_ID` and `SENTINEL_CLIENT_SECRET`; do not
paste either value into this notebook or commit the environment file.

Dates are inclusive model days. The acquisition sends the exclusive end as the
day after `end_date`, forming complete UTC daily intervals. A 90-day period is
therefore represented by a start date and a last included date 89 days later.

The configured polygon, wheat crop and rain-fed zero-irrigation management remain
teaching assumptions unless you replace them with explicitly verified metadata.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from root_zone_water.config import generate_config

# Edit these five values before a personal download.
START_DATE = "2025-05-01"
END_DATE = "2025-07-29"
FILTER_NAME = "ekf"                 # "ekf", "ukf", or "open_loop"
DATASET_PATH = "data/my-root-zone-dataset"
CONFIG_PATH = "config_my_dataset.json"
ENV_FILE = ".env"                   # private file containing the CDSE values

custom_config = generate_config(
    start_date=START_DATE,
    end_date=END_DATE,
    filter_name=FILTER_NAME,
    dataset_path=DATASET_PATH,
    output_path=ROOT / CONFIG_PATH,
)
display({"config_path": CONFIG_PATH,
         "dataset_path": custom_config["cache_dir"],
         "start_date": custom_config["start_date"],
         "end_date": custom_config["end_date"],
         "filter": custom_config["filter"],
         "mode": custom_config["mode"]})

{'config_path': 'config_my_dataset.json',
 'dataset_path': 'data/my-root-zone-dataset',
 'start_date': '2025-05-01',
 'end_date': '2025-07-29',
 'filter': 'ekf',
 'mode': 'cached'}

## Generate the dataset

The next cell is intentionally opt-in because it makes network requests and consumes
CDSE service quota. Leave `RUN_DOWNLOAD=False` while editing or testing the config.
When ready, set it to `True` and run the cell. The generated config already points
to the dataset directory, so the saved data can immediately be selected by the
estimation notebook or by a command-line run.

In [2]:
RUN_DOWNLOAD = False

if RUN_DOWNLOAD:
    from dotenv import load_dotenv
    from root_zone_water.acquisition import acquire

    load_dotenv(ROOT / ENV_FILE, override=False)
    manifest = acquire(custom_config, ROOT / DATASET_PATH)
    display({"status": manifest["status"],
             "weather_days": manifest.get("weather_days"),
             "satellite_rows": manifest.get("satellite_rows"),
             "usable_ndmi_days": manifest.get("usable_ndmi_days"),
             "errors": manifest.get("errors")})
else:
    print("Download is disabled. Set RUN_DOWNLOAD=True only when the local env file is ready.")

Download is disabled. Set RUN_DOWNLOAD=True only when the local env file is ready.


## Use the saved dataset offline

Inspect `data/my-root-zone-dataset/manifest.json` before estimating. It records
complete weather coverage, accepted/rejected satellite intervals, sample counts,
catalog candidates, requests, raw responses and processing settings.

The generated config is now the bridge between acquisition and estimation:

```powershell
python -m root_zone_water.runner --config config_my_dataset.json --output outputs/my-dataset
python execute_notebook.py
```

For the main estimation notebook, copy the generated `cache_dir`, dates and filter
from `config_my_dataset.json` into `config.json`, or run the runner with `--config`
as shown above. Cached mode does not load credentials or call APIs.

## What the filters mean

`ekf` uses the analytical state Jacobian of the water-balance transition. `ukf`
propagates scalar sigma points through that transition. `open_loop` performs prediction
only and never assimilates NDMI. All three use the same forcing, proxy conversion,
quality decisions and uncertainty configuration.

The root-zone estimate is speculative: Sentinel-2 supplies an indirect vegetation
signal, not field-measured root-zone storage. No interpolation or invented observation
is added for dates without an accepted image.